# DataMate pipeline_llm — Full End-to-End Runner trên Google Colab Pro

Notebook này cho phép bạn chạy **toàn bộ hệ thống DataMate `pipeline_llm` (Qwen3-14B + DuckDB)** trực tiếp trên máy ảo **Google Colab Pro (L4 / A100 / G4 RTX PRO 6000 Blackwell)**:
- Không cần GPU ở máy cá nhân.
- Không bị độ trễ mạng giữa Python Worker và `vLLM` (chạy qua `http://127.0.0.1:8000/v1` ngay trên Colab).
- Tự động lưu `cache/` (LLM responses, schema mappings, DuckDB canonical) và `submission.jsonl` về **Google Drive** để nếu Colab ngắt kết nối vẫn resume tức thì không mất tiến độ.

### Chuẩn bị:
1. Đóng gói thư mục project ở máy local (hoặc push lên GitHub cá nhân):
   ```bash
   zip -r datamate_project.zip pyproject.toml src eval evals rules train -x "*/__pycache__/*"
   ```
   Upload `datamate_project.zip` (và zip dữ liệu cần chạy nếu có) lên Google Drive tại `/MyDrive/zalo_aic_2026/`.
2. Chọn Runtime trên Colab Pro: **L4 GPU (24GB)**, **A100 GPU (40/80GB)**, hoặc **G4 GPU (RTX PRO 6000 96GB)**.

In [ ]:
# 1. Kiểm tra GPU trên Colab Pro
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive và giải nén project ra ổ SSD /content/work của Colab (đọc/ghi cực nhanh)
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/zalo_aic_2026")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

WORK_DIR = Path("/content/datamate")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Giải nén project từ Google Drive (hoặc git clone nếu dùng repo riêng)
PROJECT_ZIP = DRIVE_DIR / "datamate_project.zip"
if PROJECT_ZIP.is_file():
    !unzip -qo "{PROJECT_ZIP}" -d "{WORK_DIR}"
    print("Unpacked project to", WORK_DIR)
else:
    print(f"Hãy upload {PROJECT_ZIP} lên Google Drive hoặc git clone vào {WORK_DIR}")

# Liên kết thư mục cache trên Google Drive để giữ lại toàn bộ LLM cache & schema mappings qua các phiên Colab
DRIVE_CACHE = DRIVE_DIR / "cache"
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["DATAMATE_CACHE"] = str(DRIVE_CACHE)
%cd /content/datamate

In [ ]:
# 3. Cài đặt vLLM và toàn bộ thư viện của DataMate (~4-5 phút)
!pip install -q vllm "huggingface_hub[hf_transfer]" python-calamine duckdb openpyxl pyarrow pyyaml pydantic pydantic-settings httpx rapidfuzz unidecode python-dateutil
!pip install -q -e . --no-deps

In [ ]:
# 4. Khởi động vLLM Server (Qwen/Qwen3-14B-FP8) dưới background tại 127.0.0.1:8000
import subprocess
import time
import urllib.request

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
gpu_mb = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]
).split()[0])
if gpu_mb >= 40000:
    MAX_LEN, EXTRA, WORKERS = "16384", ["--gpu-memory-utilization", "0.92"], 16
else:
    MAX_LEN, EXTRA, WORKERS = "12288", ["--gpu-memory-utilization", "0.95", "--max-num-seqs", "16"], 8

cmd = [
    "vllm", "serve", "Qwen/Qwen3-14B-FP8",
    "--served-model-name", "qwen3-14b",
    "--max-model-len", MAX_LEN,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--enable-prefix-caching",
    "--reasoning-parser", "qwen3",
    *EXTRA,
]
vllm_log = open("/content/vllm.log", "w")
vllm = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
print(f"Starting vLLM (pid={vllm.pid}, VRAM={gpu_mb}MB, max_len={MAX_LEN}, recommended workers={WORKERS})...")

for _ in range(180):
    if vllm.poll() is not None:
        print(open("/content/vllm.log").read()[-4000:])
        raise RuntimeError("vLLM exited unexpectedly")
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5)
        print("vLLM Server is READY at http://127.0.0.1:8000/v1 !")
        break
    except Exception:
        time.sleep(10)

In [ ]:
# 5. (Tuỳ chọn) Đánh giá hệ thống pipeline_llm trên tập train (subset poc / holdout / all)
os.environ["LLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
os.environ["LLM_MODEL"] = "qwen3-14b"
os.environ["LLM_DISABLE_THINKING"] = "true"

!python -m evals.run_eval --split train --subset poc --system pipeline_llm --workers {WORKERS}

In [ ]:
# 6. Chạy tạo file nộp bài (submission.jsonl) cho public_test bằng PipelineSystem (pipeline_llm) và lưu thẳng về Google Drive
TARGET_SPLIT_DIR = "/content/datamate/public_test"
OUT_DIR = DRIVE_DIR / "result_pipeline"

!python -m datamate.predict \
    --data-dir "{TARGET_SPLIT_DIR}" \
    --out-dir "{OUT_DIR}" \
    --system pipeline_llm \
    --workers {WORKERS}

print("\nSubmission file saved at:", OUT_DIR / "submission.jsonl")
from google.colab import files
if (OUT_DIR / "submission.jsonl").is_file():
    files.download(str(OUT_DIR / "submission.jsonl"))

In [ ]:
# 7. Dừng vLLM khi hoàn tất (bỏ dấu # bên dưới khi muốn tắt server)
# vllm.terminate()
# print("vLLM terminated.")